<a href="https://colab.research.google.com/github/monaalfayez09-rgb/murshid_Mona_Labs_HW/blob/main/HW_Mona_Day2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import copy
import json
import re

RECORDS = {
    "REF-1001": {"owner": "demo-user-1", "status": "under_review"},
    "REF-1002": {"owner": "demo-user-2", "status": "approved"},
}


def dispatch(call, trusted_user):
    # A tool call is untrusted input. Require exactly these two fields.
    if not isinstance(call, dict) or set(call) != {"name", "arguments"}:
        return {"code": "invalid_call"}

    # Only this read-only operation is allowed.
    if call["name"] != "check_status":
        return {"code": "tool_not_allowed"}

    arguments = call["arguments"]
    if not isinstance(arguments, dict) or set(arguments) != {"reference"}:
        return {"code": "invalid_arguments"}

    reference = arguments["reference"]
    if not isinstance(reference, str) or re.fullmatch(r"REF-[0-9]{4}", reference) is None:
        return {"code": "invalid_arguments"}

    record = RECORDS.get(reference)

    # Give the same outward answer for missing and unauthorized records.
    if record is None or record["owner"] != trusted_user:
        return {"code": "unavailable"}

    return {"code": "ok", "status": record["status"]}


def run_homework():
    original_records = copy.deepcopy(RECORDS)

    cases = [
        (
            {"name": "check_status", "arguments": {"reference": "REF-1001"}},
            {"code": "ok", "status": "under_review"},
        ),
        (
            {"name": "check_status", "arguments": {"reference": "REF-1002"}},
            {"code": "unavailable"},
        ),
        (
            {"name": "check_status", "arguments": {"reference": "REF-9999"}},
            {"code": "unavailable"},
        ),
        (
            {"name": "delete_record", "arguments": {"reference": "REF-1001"}},
            {"code": "tool_not_allowed"},
        ),
        (
            {
                "name": "check_status",
                "arguments": {
                    "reference": "REF-1001",
                    "owner": "demo-user-2",
                },
            },
            {"code": "invalid_arguments"},
        ),
        (
            {"name": "check_status", "arguments": {"reference": 1001}},
            {"code": "invalid_arguments"},
        ),
        (
            "not a tool object",
            {"code": "invalid_call"},
        ),
    ]

    results = []
    for number, (call, expected) in enumerate(cases, start=1):
        actual = dispatch(call, trusted_user="demo-user-1")
        results.append({
            "case": number,
            "result": actual,
            "passed": actual == expected,
        })

    report = {
        "mode": "simulated_tool_protocol",
        "passed": sum(row["passed"] for row in results),
        "total": len(results),
        "records_unchanged": RECORDS == original_records,
        "cases": results,
    }

    print(json.dumps(report, indent=2))
    return report

In [3]:
report = run_homework()

assert report["passed"] == 7
assert report["records_unchanged"] is True

{
  "mode": "simulated_tool_protocol",
  "passed": 7,
  "total": 7,
  "records_unchanged": true,
  "cases": [
    {
      "case": 1,
      "result": {
        "code": "ok",
        "status": "under_review"
      },
      "passed": true
    },
    {
      "case": 2,
      "result": {
        "code": "unavailable"
      },
      "passed": true
    },
    {
      "case": 3,
      "result": {
        "code": "unavailable"
      },
      "passed": true
    },
    {
      "case": 4,
      "result": {
        "code": "tool_not_allowed"
      },
      "passed": true
    },
    {
      "case": 5,
      "result": {
        "code": "invalid_arguments"
      },
      "passed": true
    },
    {
      "case": 6,
      "result": {
        "code": "invalid_arguments"
      },
      "passed": true
    },
    {
      "case": 7,
      "result": {
        "code": "invalid_call"
      },
      "passed": true
    }
  ]
}


Validation checks whether the call uses the allowed tool and has a correctly shaped, correctly typed reference. Authorization happens after validation: it checks whether the trusted demo user owns the matching record. An owner value inside the call cannot grant access because the user could write that value themselves. The records are synthetic, the demo identity is not real authentication, and these calls are supplied by hand rather than selected by a language model.